In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('dirty_cafe_sales.csv')
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [7]:
# Cleaning column names: removing spaces and converting to lowercase
df.columns = [col.strip().replace(' ', '_').lower() for col in df.columns]

# Rename specifically to match Pythonic naming conventions if needed
# From: 'Price Per Unit' -> 'price_per_unit'
print(df.columns)

Index(['transaction_id', 'item', 'quantity', 'price_per_unit', 'total_spent',
       'payment_method', 'location', 'transaction_date'],
      dtype='str')


Standardization: Python developers prefer snake_case (e.g., transaction_id) because it allows for attribute-style access (like df.transaction_id) and prevents errors caused by leading/trailing spaces in the CSV heade

read_csv is used to convert the raw CSV file into a DataFrame for easy analysis

In [9]:
# Identify missing values
print(df.isnull().sum())

# Fill missing 'price_per_unit' with the median to avoid outliers
df['price_per_unit'] = pd.to_numeric(df['price_per_unit'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['total_spent'] = pd.to_numeric(df['total_spent'], errors='coerce')

# Verify that the types are now float or int
print(df.dtypes)

transaction_id         0
item                 333
quantity             138
price_per_unit       179
total_spent          173
payment_method      2579
location            3265
transaction_date     159
dtype: int64
transaction_id          str
item                    str
quantity            float64
price_per_unit      float64
total_spent         float64
payment_method          str
location                str
transaction_date        str
dtype: object


 isnull().sum() helps identify the scale of data loss.
 Using the Median for Price is safer than the Mean because it is less affected by outliers (extreme prices).

pd.to_numeric: This is essential because raw CSV data often contains "dirty" strings (like '10.5$' or 'N/A').

errors='coerce': This parameter is crucial; it converts any unparseable string into NaN (Not a Number), allowing mathematical functions like .median() to run without crashing the script.

In [ ]:
# Convert 'transaction_date' to proper datetime objects
df['transaction_date'] = pd.to_datetime(df['transaction_date'], errors='coerce')

df['item'] = df['item'].str.strip().str.lower()
df['payment_method'] = df['payment_method'].str.strip().str.capitalize()

pd.to_datetime is essential to perform time-based analysis


In [15]:
df['item'] = df['item'].str.lower().str.strip()

df['transaction_id'] = df['transaction_id'].astype(str).str.replace(r'[^a-zA-Z0-9]', '', regex=True)

str.lower() and str.strip() are used to unify categories (e.g., treating "Coffee" and " coffee " as the same item).
 str.replace with regex cleans IDs from accidental symbols or typos

In [16]:
duplicate_count = df.duplicated().sum()
df.drop_duplicates(inplace=True)
print(f"Removed {duplicate_count} duplicate rows.")

Removed 0 duplicate rows.


drop_duplicates ensures that each sale is only counted once, preventing artificial inflation of total revenue

In [17]:
df.to_csv('cleaned_cafe_sales.csv', index=False)

index=False is used so that pandas doesn't add an extra unnecessary "Unnamed: 0" column to our final clean file.